In [7]:
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from imblearn.over_sampling import SMOTE

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv


In [24]:
# ============================================================
# SECTION 1: DATASET CREATION (SYNTHETIC FLOOD DATA)
# ============================================================

# Fix random seed for reproducibility
np.random.seed(42)

# Total number of samples (areas)
num_samples = 1000

# Generate synthetic features
rainfall = np.random.uniform(50, 200, num_samples)     # Rainfall in cm
elevation = np.random.uniform(1, 100, num_samples)     # Elevation in meters
land_use = np.random.randint(0, 4, num_samples)        # Encoded land-use type

# Initialize flood labels (0 = No Flood, 1 = Flood)
flood_label = np.zeros(num_samples, dtype=int)

# Apply flood decision rule
for i in range(num_samples):
    if (rainfall[i] > 120 and elevation[i] < 20) or (rainfall[i] > 150 and land_use[i] >= 2):
        flood_label[i] = 1
    else:
        flood_label[i] = 0

# Create DataFrame
dataframe = pd.DataFrame({
    "rainfall": rainfall,
    "elevation": elevation,
    "land_use": land_use,
    "flood": flood_label
})

# Preview dataset
print(dataframe.head())


     rainfall  elevation  land_use  flood
0  106.181018  19.328160         3      0
1  192.607146  54.648194         2      1
2  159.799091  87.421638         2      1
3  139.798773  73.490264         2      0
4   73.402796  80.849554         3      0


In [25]:
# ============================================================
# SECTION 2: TRAIN–TEST SPLIT AND SMOTE BALANCING
# ============================================================

# Separate features and target
features = dataframe[["rainfall", "elevation", "land_use"]]
target = dataframe["flood"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

# ----- Class distribution BEFORE SMOTE -----
print("Class distribution BEFORE SMOTE (Training Data):")
print(y_train.value_counts())

# Apply SMOTE only on training data
smote = SMOTE(random_state=42,k_neighbors=3)
X_train, y_train = smote.fit_resample(X_train, y_train)

# ----- Class distribution AFTER SMOTE -----
print("\nClass distribution AFTER SMOTE (Training Data):")
print(y_train.value_counts())


Class distribution BEFORE SMOTE (Training Data):
flood
0    611
1    189
Name: count, dtype: int64

Class distribution AFTER SMOTE (Training Data):
flood
0    611
1    611
Name: count, dtype: int64


In [13]:
# ============================================================
# SECTION 3: GRAPH CONSTRUCTION USING KNN
# ============================================================

# ------------------------------------------------------------
# Step 1: Combine training and testing data into ONE dataset
# (GNN requires a single graph containing all nodes)
# ------------------------------------------------------------

combined_features_list = []

# Add training samples first
for i in range(len(X_train)):
    combined_features_list.append(X_train.iloc[i].values)

# Add testing samples next
for i in range(len(X_test)):
    combined_features_list.append(X_test.iloc[i].values)

# Convert combined features to NumPy array
all_node_features = np.array(combined_features_list)


combined_labels_list = []

# Add training labels first
for i in range(len(y_train)):
    combined_labels_list.append(y_train.iloc[i])

# Add testing labels next
for i in range(len(y_test)):
    combined_labels_list.append(y_test.iloc[i])

# Convert combined labels to NumPy array
all_node_labels = np.array(combined_labels_list)


# ------------------------------------------------------------
# Step 2: Create graph edges using K-Nearest Neighbors (KNN)
# ------------------------------------------------------------

# Each node will be connected to its 5 nearest neighbors
edge_index = kneighbors_graph(
    all_node_features,
    n_neighbors=5,
    include_self=False
).nonzero()

# Convert edge information to NumPy array
edge_index = np.array(edge_index)


In [26]:
# ============================================================
# SECTION 4: CREATE GRAPH DATA OBJECT
# ============================================================

# ------------------------------------------------------------
# Step 1: Convert NumPy arrays to PyTorch tensors
# ------------------------------------------------------------

# Node feature matrix (each row = one node / area)
node_features = torch.tensor(
    all_node_features,
    dtype=torch.float
)

# Node labels (flood or no flood)
node_labels = torch.tensor(
    all_node_labels,
    dtype=torch.long
)

# Graph connectivity (edges between nodes)
edge_index_tensor = torch.tensor(
    edge_index,
    dtype=torch.long
)


# ------------------------------------------------------------
# Step 2: Wrap everything into a Data object
# ------------------------------------------------------------

graph_data = Data(
    x=node_features,          # Node features
    edge_index=edge_index_tensor,  # Graph edges
    y=node_labels             # Labels
)

# NOTE:
# - We are NOT using GPU
# - No masking is used
# - PyTorch runs everything on CPU automatically
# - Data object is required for all GNN operations


In [31]:
class FloodGNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gcn_layer1 = GCNConv(3, 32)
        self.gcn_layer2 = GCNConv(32, 16)
        self.gcn_layer3 = GCNConv(16, 2)

    def forward(self, data):
        x = self.gcn_layer1(data.x, data.edge_index)
        x = F.relu(x)
        x = self.gcn_layer2(x, data.edge_index)
        x = F.relu(x)
        x = self.gcn_layer3(x, data.edge_index)
        return x


In [32]:
# ============================================================
# SECTION 6: TRAIN THE GNN MODEL
# ============================================================

# Initialize the model
gnn_model = FloodGNN()

# Optimizer to update model parameters
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.01)

# Loss function for classification
loss_function = torch.nn.CrossEntropyLoss()

# Number of training epochs
num_epochs = 2000

for epoch in range(num_epochs):
    gnn_model.train()

    # Clear old gradients
    optimizer.zero_grad()

    # Forward pass through the graph
    output = gnn_model(graph_data)

    # Compute loss using true labels
    loss = loss_function(output, graph_data.y)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Print loss occasionally
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")


Epoch 0, Loss: 19.901193618774414
Epoch 20, Loss: 1.055092453956604
Epoch 40, Loss: 0.4016743302345276
Epoch 60, Loss: 0.3462366461753845
Epoch 80, Loss: 0.32651013135910034
Epoch 100, Loss: 0.31045499444007874
Epoch 120, Loss: 0.3001554608345032
Epoch 140, Loss: 0.292261004447937
Epoch 160, Loss: 0.28601330518722534
Epoch 180, Loss: 0.2808660566806793
Epoch 200, Loss: 0.2787933051586151
Epoch 220, Loss: 0.2780038118362427
Epoch 240, Loss: 0.270084023475647
Epoch 260, Loss: 0.26675334572792053
Epoch 280, Loss: 0.26449310779571533
Epoch 300, Loss: 0.26189127564430237
Epoch 320, Loss: 0.25915348529815674
Epoch 340, Loss: 0.2569234371185303
Epoch 360, Loss: 0.25483983755111694
Epoch 380, Loss: 0.2591153383255005
Epoch 400, Loss: 0.2525291442871094
Epoch 420, Loss: 0.256446897983551
Epoch 440, Loss: 0.24840180575847626
Epoch 460, Loss: 0.24558590352535248
Epoch 480, Loss: 0.24435240030288696
Epoch 500, Loss: 0.30666446685791016
Epoch 520, Loss: 0.2699981927871704
Epoch 540, Loss: 0.2484960

In [33]:
# ============================================================
# SECTION 7: EVALUATE THE MODEL
# ============================================================

gnn_model.eval()

# Get final predictions
final_output = gnn_model(graph_data)

# Convert logits to predicted class labels
predicted_labels = final_output.argmax(dim=1)

# Calculate evaluation metrics
accuracy = accuracy_score(
    all_node_labels,
    predicted_labels.detach().numpy()
)

print("Accuracy:", accuracy)
print("Confusion Matrix:")
print(confusion_matrix(all_node_labels, predicted_labels.detach().numpy()))
print("Classification Report:")
print(classification_report(all_node_labels, predicted_labels.detach().numpy()))


Accuracy: 0.9077809798270894
Confusion Matrix:
[[352  22]
 [ 42 278]]
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92       374
           1       0.93      0.87      0.90       320

    accuracy                           0.91       694
   macro avg       0.91      0.90      0.91       694
weighted avg       0.91      0.91      0.91       694



In [35]:
torch.save(gnn_model.state_dict(), "flood_gnn_model.pth")
